# Módulo flujo: indicadores sobre movimiento de causas
Las mismas fórmulas del CEJA que `metricas`, aplicadas a las tablas del capítulo 9 del Anuario, que publican por materia,
ámbito, departamento y año las cinco cantidades del movimiento: `pendientes_inicio`, `ingresadas`, `atendidas`,
`resueltas` y `pendientes_fin`.

| indicador | fórmula |
|---|---|
| tasa de resolución (TR) | resueltas / ingresadas del año |
| tasa de congestión (TC) | atendidas / resueltas = (ingresadas + pendientes al inicio) / resueltas |
| tasa de pendencia | pendientes_fin / atendidas |
| duración estimada (días) | pendientes_fin / resueltas × 365 (estimador propio, no es del CEJA) |
| variación del stock | pendientes_fin − pendientes_inicio (dentro del año) |

**Rótulos de materia.** En el Anuario los rótulos de anticorrupción y de violencia contra la mujer están invertidos entre
cuadros: el mismo número aparece con un nombre u otro (ver `decisiones.html`). Donde se comprobó el conflicto (sentencia y
tribunales de sentencia) los dos rótulos se agrupan en un par, que no depende de cuál es cuál. En las series por año
también se agrupa el par de instrucción, porque el cuadro de capitales intercambia ahí los rótulos en 2020.
`GRUPOS_MATERIA` es un mapa cerrado: un rótulo nuevo detiene el cálculo en lugar de asignarse a ningún grupo.

In [ ]:
import numpy as np
import pandas as pd

COLUMNAS_FLUJO = ["pendientes_inicio", "ingresadas", "atendidas", "resueltas", "pendientes_fin"]

NO_PENAL = "no penal"
PENAL = "penal"
EJECUCION = "ejecución penal"

PAR_SENTENCIA = "Sentencia: anticorrupción + violencia"
PAR_TRIBUNAL = "Tribunal de sentencia: anticorrupción + violencia"
PAR_INSTRUCCION = "Instrucción: anticorrupción + violencia"

# Rótulo tal como aparece en cualquier cuadro del capítulo 9 -> (grupo, tipo de materia).
GRUPOS_MATERIA = {
    "Público Civil y Comercial": ("Civil y Comercial", NO_PENAL),
    "Público de Familia": ("Familia", NO_PENAL),
    "Público Niñez y Adolescencia": ("Niñez y Adolescencia", NO_PENAL),
    "Partido de Trabajo y Seguridad Social": ("Trabajo y Seguridad Social", NO_PENAL),
    "Partido Administrativo Coactivo Fiscal y Tributario": ("Coactivo Fiscal y Tributario", NO_PENAL),
    "Instrucción Penal": ("Instrucción penal", PENAL),
    "Instrucción Anticorrupción": ("Instrucción anticorrupción", PENAL),
    "Instrucción Contra la Violencia hacia las Mujeres": ("Instrucción violencia contra la mujer", PENAL),
    "Sentencia Penal": ("Sentencia penal", PENAL),
    "SENTENCIA ANTICORRUPCIÓN": (PAR_SENTENCIA, PENAL),
    "SENTENCIA CONTRA LA VIOLENCIA HACIA LA MUJER": (PAR_SENTENCIA, PENAL),
    "Sentencia Anticorrupción": (PAR_SENTENCIA, PENAL),
    "Sentencia Violencia C M.": (PAR_SENTENCIA, PENAL),
    "Sentencia Violencia Contra la Violencia hacia las Mujeres": (PAR_SENTENCIA, PENAL),
    "TRIBUNAL DE SENTENCIA ANTICORRUPCIÓN": (PAR_TRIBUNAL, PENAL),
    "TRIBUNAL DE SENTENCIA CONTRA LA VIOLENCIA HACIA LA MUJER": (PAR_TRIBUNAL, PENAL),
    "Tribunales de Sentencia Anticorrupción": (PAR_TRIBUNAL, PENAL),
    "Tribunales de Sentencia Contra la Violencia hacia las Mujeres": (PAR_TRIBUNAL, PENAL),
    "Tribunales de Sentencia Penal": ("Tribunales de sentencia penal", PENAL),
    "Ejecución Penal": ("Ejecución penal", EJECUCION),
}

# Para las series por año el par de instrucción también se agrupa.
GRUPOS_MATERIA_SERIE = dict(GRUPOS_MATERIA)
GRUPOS_MATERIA_SERIE["Instrucción Anticorrupción"] = (PAR_INSTRUCCION, PENAL)
GRUPOS_MATERIA_SERIE["Instrucción Contra la Violencia hacia las Mujeres"] = (PAR_INSTRUCCION, PENAL)


def asignar_grupo(df, mapa=None, columna="materia_homologada"):
    # Mapa cerrado: un rótulo sin grupo detiene el cálculo.
    if mapa is None:
        mapa = GRUPOS_MATERIA
    desconocidos = sorted(set(df[columna].dropna()) - set(mapa))
    assert not desconocidos, "rótulos de materia sin grupo: " + str(desconocidos)
    res = df.copy()
    res["grupo_materia"] = res[columna].map(lambda r: mapa[r][0])
    res["tipo_materia"] = res[columna].map(lambda r: mapa[r][1])
    return res


def miles(n, signo=False):
    # 12345 -> "12.345" (separador de miles del proyecto); con signo=True agrega + o −.
    texto = "{:,}".format(int(round(abs(n)))).replace(",", ".")
    if signo:
        return ("+" if n >= 0 else "−") + texto
    return ("−" if n < 0 else "") + texto


def dividir(numerador, denominador):
    # Divide solo donde el denominador es > 0; en el resto queda nulo.
    resultado = pd.Series(np.nan, index=numerador.index, dtype=float)
    validos = denominador > 0
    resultado[validos] = numerador[validos] / denominador[validos]
    return resultado


def banda_resolucion(x):
    # Umbrales del CEJA 2025 (tabla 4) para la tasa de resolución.
    if pd.isna(x):
        return None
    if x >= 1.0:
        return "óptima"
    if x >= 0.90:
        return "mejorable"
    return "crítica"


def banda_congestion(x):
    # Umbrales del CEJA 2025 (tabla 5) para la tasa de congestión.
    if pd.isna(x):
        return None
    if x <= 1.0:
        return "sin congestión"
    if x <= 1.5:
        return "aceptable"
    if x <= 2.0:
        return "en riesgo"
    return "crítica"


def indicadores_flujo(df):
    res = df.copy()
    ingresadas = res["ingresadas"].astype(float)
    atendidas = res["atendidas"].astype(float)
    resueltas = res["resueltas"].astype(float)
    inicio = res["pendientes_inicio"].astype(float)
    fin = res["pendientes_fin"].astype(float)

    res["tasa_resolucion"] = dividir(resueltas, ingresadas)
    res["tasa_congestion"] = dividir(atendidas, resueltas)
    res["tasa_pendencia"] = dividir(fin, atendidas)
    res["duracion_estimada_dias"] = dividir(fin, resueltas) * 365.0
    res["variacion_stock"] = fin - inicio
    res["variacion_stock_pct"] = dividir(fin - inicio, inicio) * 100.0
    res["banda_resolucion_ceja"] = res["tasa_resolucion"].map(banda_resolucion)
    res["banda_congestion_ceja"] = res["tasa_congestion"].map(banda_congestion)
    # Aritmética de capacidad, no un modelo: cuántas resoluciones más al año harían falta.
    res["resoluciones_extra_para_frenar"] = (ingresadas - resueltas).clip(lower=0)
    res["resoluciones_extra_para_vaciar_3_anios"] = res["resoluciones_extra_para_frenar"] + fin / 3.0
    res["pct_extra_para_vaciar_3_anios"] = dividir(res["resoluciones_extra_para_vaciar_3_anios"], resueltas) * 100.0
    return res
